# Module 4: Concept Erasure
**CAP6412 — Bias & Safety Auditor for T2I Models**

Fine-tunes cross-attention K/V weights to make Stable Diffusion forget a target concept.
Inspired by Forget-Me-Not (CVPR Workshop 2024).

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import torch
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path

from src.concept_erasure import ConceptEraser

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

In [ ]:
# Choose the concept to erase
CONCEPT = 'nudity'        # try: 'violence', 'in the style of Van Gogh', etc.
ANCHOR  = 'a person'

eraser = ConceptEraser(model_id='runwayml/stable-diffusion-v1-5', device=device)
print('Model loaded.')

In [ ]:
# Generate before-erasure reference images
before_dir = '../results/erasure_comparison/before'
print(f'Generating reference images for: "{CONCEPT}"')
before_imgs = eraser.verify_erasure(CONCEPT, n_images=5, output_dir=before_dir)
before_sim  = eraser.compute_clip_similarity(before_imgs, CONCEPT)
print(f'CLIP similarity BEFORE erasure: {before_sim:.4f}')

In [ ]:
# Show before images
fig, axes = plt.subplots(1, len(before_imgs), figsize=(15, 3))
for ax, img in zip(axes, before_imgs):
    ax.imshow(img)
    ax.axis('off')
fig.suptitle(f'BEFORE erasure — prompt: "{CONCEPT}"', fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# Run concept erasure
# Reduce n_steps for a quick demo; use 200+ for full erasure
eraser.erase_concept(
    concept=CONCEPT,
    anchor_concept=ANCHOR,
    n_steps=100,   # increase to 200 for full training
    lr=1e-5,
    save_dir='../models/erased_unet',
)

In [ ]:
# Generate after-erasure images
after_dir = '../results/erasure_comparison/after'
after_imgs = eraser.verify_erasure(CONCEPT, n_images=5, output_dir=after_dir)
after_sim  = eraser.compute_clip_similarity(after_imgs, CONCEPT)
print(f'CLIP similarity AFTER erasure:  {after_sim:.4f}')
reduction  = (before_sim - after_sim) / (before_sim + 1e-9) * 100
print(f'Similarity reduction:           {reduction:.1f}%')

In [ ]:
# Before vs After comparison
fig, axes = plt.subplots(2, 5, figsize=(18, 7))
for ax, img in zip(axes[0], before_imgs):
    ax.imshow(img)
    ax.axis('off')
    ax.set_title('Before', fontsize=9)
for ax, img in zip(axes[1], after_imgs):
    ax.imshow(img)
    ax.axis('off')
    ax.set_title('After', fontsize=9)
fig.suptitle(f'Concept Erasure: "{CONCEPT}" | CLIP sim: {before_sim:.3f} → {after_sim:.3f} ({reduction:.1f}% reduction)',
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/charts/erasure_comparison.png', dpi=150)
plt.show()

In [ ]:
# Verify collateral damage: unrelated prompt should be unaffected
test_prompts = ['a photo of a park', 'a photo of a mountain']
for p in test_prompts:
    imgs = eraser.verify_erasure(p, n_images=2)
    sim  = eraser.compute_clip_similarity(imgs, p)
    print(f'  CLIP sim for "{p}": {sim:.4f}  (should stay high)')